# Ajuste fino: pedestre com cadeira de rodas (Google Colab)

**Para que serve:** o YOLO-World, usado sem treino no notebook 07, reconhece a cadeira de rodas só em parte (recall de
0,64 a 0,73 nas 58 fotos de avaliação) e não sabe se há alguém sentado nela. Aqui treinamos um detector pequeno
(**YOLO11s**) em que a caixa cobre **a pessoa junto com a cadeira**, em duas classes:

| id | classe | no dataset original |
|---|---|---|
| 0 | `pedestre` | `people` |
| 1 | `pedestre_cadeira_rodas` | `peopleWithWheelchair` |

**Dados:** Roboflow Universe, projeto *crooswalk/wheelchair-tmzbn*, versão 2, licença **CC BY 4.0**
(https://universe.roboflow.com/crooswalk/wheelchair-tmzbn/dataset/2): 512 imagens (359 treino, 103 validação,
50 teste). As imagens foram coletadas da internet por quem montou o dataset; usamos para fins acadêmicos, com crédito,
e **não as versionamos** no Git.

**Conferências feitas antes do treino** (no computador local):
- nenhuma das 58 fotos de avaliação do notebook 07 aparece no dataset (hash perceptual), então a comparação
  "antes × depois" não é contaminada;
- 8 das 503 caixas de `people` têm uma cadeira de rodas dentro (1,6%): ruído pequeno, em parte acompanhantes que
  empurram a cadeira.

**Limitação:** só cadeira de rodas. Andador, muleta e bengala ficam para um próximo conjunto de dados.

**Como rodar:** Ambiente de execução → Alterar o tipo → **GPU (T4)**. Depois, execute as células em ordem. Na célula de
upload, envie o arquivo `wheelchair.v2i.yolov8.zip` (exportado do Roboflow em formato YOLOv8). No fim, o navegador
baixa `pcd_yolo11s.pt`: coloque-o em `1TIAO/GLOBAL-SOLUTION-2/models/` (pesos não vão para o Git).

In [ ]:
!pip -q install ultralytics==8.4.211
import torch, ultralytics
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (mude o ambiente para GPU)")
print("ultralytics", ultralytics.__version__)

## 1. Dados

In [ ]:
from google.colab import files
import zipfile, pathlib
enviado = files.upload()                       # escolha wheelchair.v2i.yolov8.zip
zip_nome = next(iter(enviado))
DADOS = pathlib.Path("/content/dados")
with zipfile.ZipFile(zip_nome) as z:
    z.extractall(DADOS)
for p in ("train", "valid", "test"):
    print(p, len(list((DADOS / p / "images").glob("*"))), "imagens")

In [ ]:
# data.yaml com caminhos absolutos e nomes em português (os ids 0 e 1 não mudam)
import yaml
cfg = {"path": str(DADOS), "train": "train/images", "val": "valid/images", "test": "test/images",
       "names": {0: "pedestre", 1: "pedestre_cadeira_rodas"}}
(DADOS / "data_pt.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True))
print((DADOS / "data_pt.yaml").read_text())

## 2. Treino

- **Modelo de partida:** `yolo11s.pt` (pré-treinado no COCO, que já sabe o que é pessoa).
- **640 px, até 80 épocas, parada antecipada** se a validação não melhorar em 20 épocas, semente fixa (0).
- Aumentos de dados padrão do ultralytics (espelhamento horizontal, mosaico, variação de cor), que ajudam com
  poucas imagens.

In [ ]:
from ultralytics import YOLO
modelo = YOLO("yolo11s.pt")
res = modelo.train(data=str(DADOS / "data_pt.yaml"), imgsz=640, epochs=80, patience=20, batch=16,
                   seed=0, deterministic=True, project="/content/treino", name="pcd_yolo11s", exist_ok=True)

## 3. Avaliação no conjunto de teste (50 imagens que o treino não viu)

In [ ]:
melhor = YOLO("/content/treino/pcd_yolo11s/weights/best.pt")
m = melhor.val(data=str(DADOS / "data_pt.yaml"), split="test", imgsz=640)
import pandas as pd
nomes = melhor.names
tab = pd.DataFrame({"classe": [nomes[i] for i in range(len(nomes))],
                    "precisão": m.box.p, "recall": m.box.r, "mAP50": m.box.ap50, "mAP50-95": m.box.ap})
display(tab.round(3))

In [ ]:
from IPython.display import Image, display
for f in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    p = pathlib.Path("/content/treino/pcd_yolo11s") / f
    if p.exists(): display(Image(str(p), width=900))

## 4. Baixar os pesos

Coloque `pcd_yolo11s.pt` em `1TIAO/GLOBAL-SOLUTION-2/models/`. A comparação com o YOLO-World nas 58 fotos de
avaliação e a integração ao sistema são feitas no computador local (notebook 07).

In [ ]:
import shutil
shutil.copy("/content/treino/pcd_yolo11s/weights/best.pt", "/content/pcd_yolo11s.pt")
tab.to_csv("/content/08_metricas_teste.csv", sep=";", index=False)
files.download("/content/pcd_yolo11s.pt")
files.download("/content/08_metricas_teste.csv")